# Week 2.3 Notebook: Diagnostics And Probe Exports

This notebook proves that the loader can be inspected before training. It creates metadata summaries, a batch contact sheet, frame-difference maps, motion-energy histograms, low-motion and high-motion examples, dummy probe exports, and the Week 2 gate. On real data, diagnostics should run on HAIC or Sherlock compute resources and write only to ignored diagnostics and probe-export directories.


## SSH And Batch Submission For Diagnostics

Diagnostics generate media and tables, so run real-data diagnostics through HAIC or Sherlock rather than on your laptop. Read the prompt labels as locations, and do not type the labels themselves:

| Example | Where you should be | Meaning |
| --- | --- | --- |
| `laptop$ ssh <sunetid>@<haic-login-host>` | Any directory on your laptop | Open a HAIC or Sherlock login-node terminal. Use `sherlock.stanford.edu` when your instructions say to use Sherlock. |
| `haic-login$ cd "$CODY_JEPA_ROOT"` | Login node | Move to the repo on the cluster before submitting the diagnostics script. |
| `haic-login$ sbatch scripts/gaitlu_diag.sbatch` | `$CODY_JEPA_ROOT` on the login node | Submit diagnostics to run on a compute node. |
| `haic-login$ squeue -u "$USER"` | Any directory on the login node | Monitor whether the job is waiting or running. |
| `haic-login$ sacct -j <job_id> ...` | Any directory on the login node | Inspect the completed diagnostics job. |

Keep `GAITLU_MAX_DIAGNOSTIC_ROWS` small at first. It limits the number of real sequences used for visual diagnostics and dummy probe exports. Before saving the batch script, run `mkdir -p scripts logs` from `$CODY_JEPA_ROOT` on the login node. Do not commit contact sheets, frame grids, full manifests with sensitive paths, or latent exports.


## Path Setup

This cell group keeps diagnostics and dummy exports under ignored paths. On HAIC or Sherlock, real diagnostic outputs should go to `$GAITLU_DIAGNOSTICS_DIR`, and dummy or real probe exports should go to `$GAITLU_PROBE_EXPORT_DIR`. These should usually be under `$SCRATCH` or approved project storage.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import pickle
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing pyproject.toml")


REPO_ROOT = find_repo_root()
GAITLU_ROOT = Path(os.environ.get("GAITLU_ROOT", REPO_ROOT / "data" / "gaitlu-1m"))
ARCHIVE_DIR = Path(os.environ.get("GAITLU_ARCHIVE_DIR", GAITLU_ROOT / "archives"))
RAW_DIR = Path(os.environ.get("GAITLU_EXTRACTED_DIR", GAITLU_ROOT / "raw"))
NOTEBOOK_MODE = os.environ.get("GAITLU_NOTEBOOK_MODE", "synthetic").lower()
if NOTEBOOK_MODE not in {"synthetic", "real"}:
    raise ValueError("GAITLU_NOTEBOOK_MODE must be 'synthetic' or 'real'")
SYNTHETIC_RAW_DIR = Path(os.environ.get("GAITLU_SYNTHETIC_RAW_DIR", GAITLU_ROOT / "synthetic_raw"))
MANIFEST_DIR = Path(os.environ.get("GAITLU_MANIFEST_DIR", GAITLU_ROOT / "manifests"))
DIAGNOSTICS_DIR = Path(os.environ.get("GAITLU_DIAGNOSTICS_DIR", GAITLU_ROOT / "diagnostics"))
PROBE_EXPORT_DIR = Path(os.environ.get("GAITLU_PROBE_EXPORT_DIR", GAITLU_ROOT / "probe_exports"))

for path in (ARCHIVE_DIR, MANIFEST_DIR, DIAGNOSTICS_DIR, PROBE_EXPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)
if NOTEBOOK_MODE == "synthetic":
    SYNTHETIC_RAW_DIR.mkdir(parents=True, exist_ok=True)
elif not RAW_DIR.exists():
    raise FileNotFoundError(f"GAITLU_EXTRACTED_DIR does not exist: {RAW_DIR}")

print("repo", REPO_ROOT)
print("gaitlu_root", GAITLU_ROOT)
print("notebook_mode", NOTEBOOK_MODE)
print("raw_dir", RAW_DIR)
print("synthetic_raw_dir", SYNTHETIC_RAW_DIR)


## Synthetic Data, Index, And Loader

This cell group rebuilds the synthetic fixture, index, splits, and validation batch used by diagnostics.


In [ ]:
def stable_int(text, seed=0):
    payload = f"{seed}:{text}".encode("utf-8")
    return int(hashlib.sha1(payload).hexdigest()[:16], 16)


def create_synthetic_gaitlu_fixture(raw_dir=None, variant="anonymized_sil"):
    if NOTEBOOK_MODE != "synthetic":
        raise RuntimeError("Refusing to create synthetic fixtures when GAITLU_NOTEBOOK_MODE is not 'synthetic'")
    raw_dir = SYNTHETIC_RAW_DIR if raw_dir is None else Path(raw_dir)
    root = raw_dir / variant
    root.mkdir(parents=True, exist_ok=True)
    for top in range(4):
        for mid in range(3):
            leaf = root / f"{top:03d}" / f"{mid:03d}" / "000"
            leaf.mkdir(parents=True, exist_ok=True)
            path = leaf / f"{top:03d}_{mid:03d}_000.pkl"
            if path.exists():
                continue
            frames = 18 + top + mid
            height, width = 64, 44
            arr = np.zeros((frames, height, width), dtype=np.uint8)
            for t in range(frames):
                x = (2 * t + 3 * top + mid) % (width - 8)
                y = 12 + 3 * mid
                arr[t, y:y + 24, x:x + 6] = 255
                arr[t, max(0, y - 5):y, min(width - 1, x + 2):min(width, x + 9)] = 160
            with path.open("wb") as f:
                pickle.dump(arr, f)
    return root


def discover_pkl_root(base):
    base = Path(base)
    for name in ("silhouette_cut_pkl", "anonymized_sil"):
        candidate = base / name
        if candidate.exists() and any(candidate.rglob("*.pkl")):
            return candidate
    pkls = sorted(base.rglob("*.pkl"))
    if not pkls:
        raise FileNotFoundError(f"No .pkl files under {base}")
    return Path(os.path.commonpath([str(p.parent) for p in pkls]))


def coerce_sequence(payload):
    if isinstance(payload, dict):
        for key in ("silhouette", "silhouettes", "frames", "imgs", "data", "seq"):
            if key in payload:
                return coerce_sequence(payload[key])
        for value in payload.values():
            try:
                return coerce_sequence(value)
            except Exception:
                continue
        raise ValueError("Dictionary payload did not contain an array-like sequence")
    arr = np.asarray(payload)
    arr = np.squeeze(arr)
    if arr.ndim == 2:
        arr = arr[None, :, :]
    if arr.ndim == 4 and arr.shape[-1] in (1, 3, 4):
        arr = arr[..., 0]
    if arr.ndim == 4 and arr.shape[1] in (1, 3, 4):
        arr = arr[:, 0, :, :]
    if arr.ndim != 3:
        raise ValueError(f"Expected sequence shape [T,H,W], got {arr.shape}")
    return arr


def read_pickle_sequence(path):
    with Path(path).open("rb") as f:
        payload = pickle.load(f)
    return coerce_sequence(payload)


def build_index(pkl_root):
    pkl_root = Path(pkl_root)
    rows = []
    for pkl_path in sorted(pkl_root.rglob("*.pkl")):
        rel = pkl_path.relative_to(pkl_root)
        shard_tokens = rel.parts[:-1]
        row = {
            "sequence_id": hashlib.sha1(str(rel).encode("utf-8")).hexdigest()[:16],
            "relative_pkl_path": str(rel),
            "shard_0": shard_tokens[0] if len(shard_tokens) > 0 else "",
            "shard_1": shard_tokens[1] if len(shard_tokens) > 1 else "",
            "shard_2": shard_tokens[2] if len(shard_tokens) > 2 else "",
            "shard_path": str(Path(*shard_tokens)) if shard_tokens else "",
        }
        try:
            arr = read_pickle_sequence(pkl_path)
            row.update({
                "num_frames": int(arr.shape[0]),
                "height": int(arr.shape[1]),
                "width": int(arr.shape[2]),
                "dtype": str(arr.dtype),
                "read_ok": True,
                "error": "",
            })
        except Exception as exc:
            row.update({
                "num_frames": 0,
                "height": 0,
                "width": 0,
                "dtype": "",
                "read_ok": False,
                "error": f"{type(exc).__name__}: {exc}",
            })
        rows.append(row)
    return pd.DataFrame(rows)


def choose_proxy_split_key(index):
    for column in ("shard_0", "shard_1", "shard_2", "shard_path", "sequence_id"):
        values = [v for v in sorted(index[column].astype(str).unique()) if v]
        if 1 < len(values) < len(index):
            return column
    return "sequence_id"


def make_splits(index, seed=7, val_fraction=0.25):
    df = index[index["read_ok"]].copy().reset_index(drop=True)
    proxy_column = choose_proxy_split_key(df)
    df["proxy_split_key"] = df[proxy_column].astype(str)
    groups = sorted(df["proxy_split_key"].unique())
    if len(groups) == 1:
        val_groups = set()
    else:
        ranked = sorted(groups, key=lambda group: stable_int(group, seed=seed))
        n_val = max(1, min(len(groups) - 1, round(len(groups) * val_fraction)))
        val_groups = set(ranked[:n_val])
    df["split"] = np.where(df["proxy_split_key"].isin(val_groups), "val", "train")
    df["split_seed"] = seed
    df["split_proxy_column"] = proxy_column
    columns = [
        "sequence_id", "relative_pkl_path", "shard_0", "shard_1", "shard_2", "shard_path",
        "num_frames", "height", "width", "dtype", "read_ok", "error",
        "proxy_split_key", "split", "split_seed", "split_proxy_column",
    ]
    return df[columns].sort_values("sequence_id").reset_index(drop=True)


class GaitLUPklDataset(Dataset):
    def __init__(self, pkl_root, split_manifest, split, clip_length=16, seed=7):
        self.pkl_root = Path(pkl_root)
        self.rows = split_manifest[split_manifest["split"] == split].reset_index(drop=True)
        self.split = split
        self.clip_length = int(clip_length)
        self.seed = int(seed)
        self.epoch = 0
        if self.rows.empty:
            raise ValueError(f"No rows for split={split}")

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.rows)

    def _start(self, sequence_id, num_frames):
        max_start = max(0, int(num_frames) - self.clip_length)
        if max_start == 0:
            return 0
        if self.split == "train":
            rng = random.Random(f"{self.seed}:{self.epoch}:{sequence_id}")
            return rng.randrange(max_start + 1)
        return stable_int(f"val:{sequence_id}:{self.clip_length}", seed=self.seed) % (max_start + 1)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        arr = read_pickle_sequence(self.pkl_root / row["relative_pkl_path"])
        start = self._start(row["sequence_id"], arr.shape[0])
        clip = arr[start:start + self.clip_length]
        if clip.shape[0] < self.clip_length:
            pad_count = self.clip_length - clip.shape[0]
            pad = np.repeat(clip[-1:, :, :], pad_count, axis=0)
            clip = np.concatenate([clip, pad], axis=0)
        clip = clip.astype(np.float32)
        if clip.max() > 1.0:
            clip = clip / 255.0
        video = torch.from_numpy(clip).unsqueeze(1).contiguous()
        return {
            "video": video,
            "sequence_id": row["sequence_id"],
            "split": row["split"],
            "proxy_label": row["proxy_split_key"],
        }


def run_week2_gate(pkl_root, seed=7, clip_length=16, batch_size=2):
    index = build_index(pkl_root)
    split_a = make_splits(index, seed=seed)
    split_b = make_splits(index, seed=seed)
    splits_reproduce = split_a.to_csv(index=False) == split_b.to_csv(index=False)

    val_a = GaitLUPklDataset(pkl_root, split_a, "val", clip_length=clip_length, seed=seed)
    val_b = GaitLUPklDataset(pkl_root, split_b, "val", clip_length=clip_length, seed=seed)
    batch_a = next(iter(DataLoader(val_a, batch_size=batch_size, shuffle=False, num_workers=0)))
    batch_b = next(iter(DataLoader(val_b, batch_size=batch_size, shuffle=False, num_workers=0)))

    validation_reproduces = torch.equal(batch_a["video"], batch_b["video"])
    video_shape = list(batch_a["video"].shape)
    shape_ok = len(video_shape) == 5 and video_shape[1:3] == [clip_length, 1]
    result = {
        "splits_reproduce": bool(splits_reproduce),
        "validation_batches_reproduce": bool(validation_reproduces),
        "batch_shape": video_shape,
        "shape_ok": bool(shape_ok),
    }
    assert result["splits_reproduce"], result
    assert result["validation_batches_reproduce"], result
    assert result["shape_ok"], result
    return result, split_a, batch_a


In [ ]:
if NOTEBOOK_MODE == "synthetic":
    PKL_ROOT = create_synthetic_gaitlu_fixture()
else:
    PKL_ROOT = discover_pkl_root(RAW_DIR)
MAX_DIAGNOSTIC_ROWS = int(os.environ.get("GAITLU_MAX_DIAGNOSTIC_ROWS", "64"))
index = build_index(PKL_ROOT)
split_manifest = make_splits(index, seed=7)
diagnostic_manifest = pd.concat([
    split_manifest[split_manifest["split"] == "val"].head(MAX_DIAGNOSTIC_ROWS),
    split_manifest[split_manifest["split"] == "train"].head(MAX_DIAGNOSTIC_ROWS),
]).sort_values("sequence_id").reset_index(drop=True)
val_ds = GaitLUPklDataset(PKL_ROOT, diagnostic_manifest, "val", clip_length=16, seed=7)
batch = next(iter(DataLoader(val_ds, batch_size=3, shuffle=False, num_workers=0)))
print("mode", NOTEBOOK_MODE)
print("pkl_root", PKL_ROOT)
print("diagnostic_rows", len(diagnostic_manifest))
print(batch["video"].shape)


## Metadata Summaries

This cell proves that read status, dimensions, frame counts, and split counts are visible before training.


In [ ]:
summary = {
    "num_sequences": int(len(index)),
    "read_ok": int(index["read_ok"].sum()),
    "read_errors": int((~index["read_ok"]).sum()),
    "frame_min": int(index["num_frames"].min()),
    "frame_max": int(index["num_frames"].max()),
    "height_values": sorted(index["height"].unique().tolist()),
    "width_values": sorted(index["width"].unique().tolist()),
    "split_counts": split_manifest["split"].value_counts().to_dict(),
    "diagnostic_rows": int(len(diagnostic_manifest)),
}
summary_path = DIAGNOSTICS_DIR / "synthetic_metadata_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))


## Batch Contact Sheet

This cell saves a small contact sheet from one validation clip. Real participant media must stay under ignored diagnostics paths.


In [ ]:
from PIL import Image


def make_contact_sheet(video, frames=8, scale=2):
    video = video.detach().cpu()
    picks = np.linspace(0, video.shape[0] - 1, frames, dtype=int)
    images = []
    for idx in picks:
        arr = video[idx, 0].numpy()
        arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr, mode="L").resize((arr.shape[1] * scale, arr.shape[0] * scale))
        images.append(img)
    sheet = Image.new("L", (images[0].width * len(images), images[0].height))
    for offset, img in enumerate(images):
        sheet.paste(img, (offset * img.width, 0))
    return sheet

sheet = make_contact_sheet(batch["video"][0])
contact_path = DIAGNOSTICS_DIR / "synthetic_contact_sheet.png"
sheet.save(contact_path)
print(contact_path)
sheet


## Frame-Difference Map

This cell computes and saves an average adjacent-frame difference map for one clip.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

video = batch["video"][0]
diff_map = (video[1:] - video[:-1]).abs().mean(dim=0).squeeze(0).numpy()
fig, ax = plt.subplots(figsize=(4, 5))
ax.imshow(diff_map, cmap="magma")
ax.set_axis_off()
diff_path = DIAGNOSTICS_DIR / "synthetic_frame_difference.png"
fig.savefig(diff_path, bbox_inches="tight", dpi=120)
plt.close(fig)
print(diff_path)


## Motion-Energy Histogram

This cell computes motion energy per validation clip and saves a histogram.


In [ ]:
videos = batch["video"]
energies = (videos[:, 1:] - videos[:, :-1]).abs().mean(dim=(1, 2, 3, 4)).numpy()
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(energies, bins=5)
ax.set_xlabel("motion energy")
ax.set_ylabel("clips")
hist_path = DIAGNOSTICS_DIR / "synthetic_motion_energy_hist.png"
fig.savefig(hist_path, bbox_inches="tight", dpi=120)
plt.close(fig)
print({"energies": energies.tolist(), "path": str(hist_path)})


## Low-Motion And High-Motion Examples

This cell identifies examples by motion energy so broken low-motion or flickering high-motion clips are easy to audit.


In [ ]:
energy_rows = []
for i in range(len(val_ds)):
    sample = val_ds[i]
    energy = (sample["video"][1:] - sample["video"][:-1]).abs().mean().item()
    energy_rows.append({
        "sequence_id": sample["sequence_id"],
        "proxy_label": sample["proxy_label"],
        "motion_energy": energy,
    })
energy_df = pd.DataFrame(energy_rows).sort_values("motion_energy")
examples_path = DIAGNOSTICS_DIR / "synthetic_motion_examples.csv"
energy_df.to_csv(examples_path, index=False)
print("low")
print(energy_df.head(2))
print("high")
print(energy_df.tail(2))


## Dummy Probe Exports

This cell writes deterministic dummy `s_attr` and `s_dyn` vectors with sequence IDs, splits, and proxy labels. Later weeks replace these with frozen CoDy-JEPA latents.


In [ ]:
def dummy_vectors(row, pkl_root, attr_dim=4, dyn_dim=4):
    arr = read_pickle_sequence(Path(pkl_root) / row["relative_pkl_path"]).astype(np.float32)
    if arr.max() > 1:
        arr = arr / 255.0
    motion = np.abs(arr[1:] - arr[:-1]).mean() if arr.shape[0] > 1 else 0.0
    rng = np.random.default_rng(stable_int(row["sequence_id"], seed=101) % (2**32))
    s_attr = np.array([arr.mean(), arr.std(), row["height"] / 100.0, row["width"] / 100.0], dtype=np.float32)
    s_dyn = np.array([motion, row["num_frames"] / 100.0, rng.normal(0, 0.01), rng.normal(0, 0.01)], dtype=np.float32)
    return s_attr[:attr_dim], s_dyn[:dyn_dim]

export_rows = []
for _, row in diagnostic_manifest.iterrows():
    s_attr, s_dyn = dummy_vectors(row, PKL_ROOT)
    out = {"sequence_id": row["sequence_id"], "split": row["split"], "proxy_label": row["proxy_split_key"]}
    out.update({f"s_attr_{i}": float(value) for i, value in enumerate(s_attr)})
    out.update({f"s_dyn_{i}": float(value) for i, value in enumerate(s_dyn)})
    export_rows.append(out)
probe_df = pd.DataFrame(export_rows).sort_values("sequence_id")
probe_path = PROBE_EXPORT_DIR / "synthetic_dummy_probe_exports.csv"
probe_df.to_csv(probe_path, index=False)
print(probe_path)
probe_df.head()


## Week 2 Gate

The final cell repeats the gate after diagnostics and export creation.


In [ ]:
result, split_manifest_gate, batch_gate = run_week2_gate(PKL_ROOT)
print(json.dumps(result, indent=2))
assert {"sequence_id", "split", "proxy_label"}.issubset(probe_df.columns)
assert any(column.startswith("s_attr_") for column in probe_df.columns)
assert any(column.startswith("s_dyn_") for column in probe_df.columns)
